# 08 — Run All Notebooks

Optional notebook to execute the pipeline notebooks in order using `nbclient`. You can also run each notebook manually.

In [ ]:
from pathlib import Path
import nbformat
from nbclient import NotebookClient

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "config" / "genes.json").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root.")

PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
EXECUTED_DIR = NOTEBOOK_DIR / "executed"
EXECUTED_DIR.mkdir(exist_ok=True)

notebooks_to_run = [
    "00_Project_Setup_and_Validation.ipynb",
    "01_Data_Processing_Pipeline.ipynb",
    "02_Experiment_1_Standard_MLP.ipynb",
    "03_Experiment_2_Dropout_Comparison.ipynb",
    "04_Experiment_3_BatchNorm_Comparison.ipynb",
    "05_Experiment_4_Residual_MLP.ipynb",
    "06_Experiment_5_MultiBranch_MultiTask.ipynb",
    "07_Results_Excel_and_Visuals.ipynb",
]

for notebook_name in notebooks_to_run:
    path = NOTEBOOK_DIR / notebook_name
    print(f"Executing {path}")

    with open(path, "r", encoding="utf-8") as f:
        nb = nbformat.read(f, as_version=4)

    client = NotebookClient(
        nb,
        timeout=7200,
        kernel_name="python3",
        resources={"metadata": {"path": str(PROJECT_ROOT)}},
    )
    client.execute()

    out_path = EXECUTED_DIR / notebook_name
    with open(out_path, "w", encoding="utf-8") as f:
        nbformat.write(nb, f)

    print(f"Saved executed notebook to {out_path}")

print("All notebooks executed.")